In [ ]:
import re
import pandas as pd
import numpy as np
from datetime import datetime
from IPython.display import display, Markdown
from numpyro import distributions as dist
import yaml as yml

from emu_renewal.constants import BASE_PATH
from emu_renewal.inputs import get_worldbank_national_pop, get_income_group, get_fb_visited_mobility, \
    get_fb_singletile_mobility, process_raw_google_mobility, get_google_mobility, get_country_pop, \
    get_undesa_national_pop, get_country_vacc_data
from emu_renewal.outputs import get_table_df_from_priors_dict
from emu_renewal.document import get_func_blurb
from emu_renewal.indicators import get_deaths_target, get_cases_target, filter_seroprev, get_owid_hosps, \
    get_all_seroprev, get_seroprev_pooled_totals, get_var_target, extract_specific_var, get_country_vars, \
    get_alpha_info, get_delta_info, get_specific_var_props, get_ba2_target, get_ba5_target, \
    get_ba2_info, get_ba5_info
from emu_renewal.run import find_run_start_time, find_run_end_time, get_hosp_target, get_seroprev_target, \
    get_mobility_provider, run_calibration
from emu_renewal.geospatial import FacebookMobilityBuilder
from emu_renewal.renew import MultiStrainModel
from emu_renewal.distributions import GammaDens
from emu_renewal.process import _get_cos_curve_at_x
from emu_renewal.mobility import NoMobilityProvider, WeightedExpMobilityProvider, SingleSeriesExpMobilityProvider
from emu_renewal.calibration import custom_init, StandardCalib
from emu_renewal.priors import get_standard_priors

In [ ]:
txt = "# Methods\n"
txt += "## Modelled population size\n"
txt += get_func_blurb(get_country_pop)
txt += get_func_blurb(get_worldbank_national_pop)
txt += get_func_blurb(get_undesa_national_pop)
txt += "\n\n## Analysis period\n"
txt += get_func_blurb(find_run_start_time)
txt += get_func_blurb(get_country_vacc_data) + "\n\n"
txt += get_func_blurb(find_run_end_time)
txt += "\n\n## Calibration targets\n"
txt += "\n\n### WHO indicators\n"
txt += get_func_blurb(get_deaths_target)
txt += get_func_blurb(get_cases_target)
txt += "\n\n### Hospitalisations\n"
txt += get_func_blurb(get_owid_hosps)
txt += get_func_blurb(get_hosp_target) + "\n\n"
txt += "\n\n### Seroprevalence\n"
txt += get_func_blurb(get_all_seroprev)
txt += get_func_blurb(get_income_group)
txt += get_func_blurb(filter_seroprev)
txt += get_func_blurb(get_seroprev_pooled_totals) + "\n\n"
txt += get_func_blurb(get_seroprev_target) + "\n\n"
txt += "\n\n### Variants\n\n"
txt += "#### Data extraction\n"
txt += get_func_blurb(get_country_vars)
txt += get_func_blurb(extract_specific_var)
txt += get_func_blurb(get_specific_var_props) + "\n\n"
txt += get_func_blurb(get_var_target)
txt += "\n\n#### Alpha\n"
txt += get_func_blurb(get_alpha_info)
txt += "\n\n#### Delta\n"
txt += get_func_blurb(get_delta_info)
txt += "\n\n#### Omicron BA.2\n"
txt += get_func_blurb(get_ba2_target)
txt += get_func_blurb(get_ba2_info)
txt += "\n\n#### Omicron BA.5\n"
txt += get_func_blurb(get_ba5_target)
txt += get_func_blurb(get_ba5_info)
txt += "\n\n## Mobility\n"
txt += get_func_blurb(get_mobility_provider)
txt += "\n\n### No mobility\n"
no_mob_provider = NoMobilityProvider()
txt += get_func_blurb(NoMobilityProvider.__init__)
txt += "\n\n### Google mobility\n"
n_domains = 6
dummy_g_mob = pd.DataFrame(1.0, index=range(10), columns=range(n_domains))
dummy_g_priors = {"mob_weights": dist.Uniform(np.zeros(n_domains), np.ones(n_domains)), "mob_exp": None}
weighted_exp_mob_provider = WeightedExpMobilityProvider(dummy_g_mob, dummy_g_priors)
txt += get_func_blurb(process_raw_google_mobility)
txt += get_func_blurb(get_google_mobility)
txt += get_func_blurb(weighted_exp_mob_provider.__init__)
txt += get_func_blurb(weighted_exp_mob_provider.get_parameterised_mobility)
txt += "\n\n### Facebook mobility\n"
dummy_fb_mob_builder = FacebookMobilityBuilder()
txt += get_func_blurb(FacebookMobilityBuilder.build_mobility) + "\n\n"
txt += get_func_blurb(get_fb_visited_mobility)
txt += get_func_blurb(get_fb_singletile_mobility)
dummy_fb_mob = pd.Series(1.0, index=range(10))
dummy_fb_priors = {"mob_exp": None}
single_series_mob_provider = SingleSeriesExpMobilityProvider(dummy_fb_mob, dummy_fb_priors)
txt += get_func_blurb(single_series_mob_provider.__init__)
txt += get_func_blurb(single_series_mob_provider.get_parameterised_mobility)
dummy_start = datetime(2020, 1, 1)
dummy_end = datetime(2021, 12, 31)
dummy_model = MultiStrainModel(1.0, dummy_start, dummy_end, [""], dummy_start, no_mob_provider, False)
txt += "\n\n## Variable process\n"
txt += get_func_blurb(dummy_model.initialise_var_proc)
txt += get_func_blurb(_get_cos_curve_at_x)
txt += get_func_blurb(dummy_model.fit_process_curve)
txt += "\n\n## Renewal model\n"
txt += get_func_blurb(dummy_model.renew)
dummy_dist = GammaDens()
txt += get_func_blurb(dummy_dist.get_params) + "\n\n"
txt += get_func_blurb(dummy_model.renewal_func)
txt += "\n\n## Calibration\n"
txt += get_func_blurb(run_calibration)
txt += get_func_blurb(custom_init)
dummy_calib = StandardCalib(dummy_model, {}, {})
txt += get_func_blurb(dummy_calib.__init__)
txt += get_func_blurb(dummy_calib.sample_calib_params)
txt += get_func_blurb(get_standard_priors)

In [ ]:
display(Markdown(txt))

In [ ]:
loaded_priors = yml.safe_load(open(BASE_PATH / "data/config/priors.yml", "r"))

{{< pagebreak >}}

In [ ]:
col_widths = '{tbl-colwidths="[14, 8, 8, 70]"}'
dur_table = get_table_df_from_priors_dict(loaded_priors["durations"])
caption = "\n: Parameters and supporting evidence to time period priors. "
Markdown(dur_table.to_markdown() + "\n" + caption + col_widths)

{{< pagebreak >}}
# References